<a href="https://colab.research.google.com/github/vkshadoww/114-2-Programing-Language/blob/main/Copy_of_HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
pip install -q google-generativeai google-auth --upgrade

In [3]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [5]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make smart decisions.


In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1pA05rdBicdtbP1LZQIQrls8HxSqm74JgKmyFKAqLvzw/edit?usp=drivesdk"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [7]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [8]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 暫時註解掉這個 model = genai.GenerativeModel('gemini-1.5-flash')

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [9]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：德文
請輸入 德文 的成績：99
已記錄：日期: 2026-04-01, 科目: 德文, 成績: 99

請輸入科目（或輸入 'q' 停止）：日文
請輸入 日文 的成績：98
已記錄：日期: 2026-04-01, 科目: 日文, 成績: 98

請輸入科目（或輸入 'q' 停止）：q


In [10]:
new_grades

[['2026-04-01', '德文', 99], ['2026-04-01', '日文', 98]]

In [11]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份基於您提供的成績數據所做的簡單摘要與常見迷思整理，全程不對學生表現進行評價。\n\n---\n\n### 成績摘要\n\n根據您提供的資料，該學生在2026年4月1日進行了以下科目的考試，並取得高分：\n\n*   **德文：** 99分\n*   **日文：** 98分\n\n**總結來說：**\n這兩科的成績均非常接近滿分，顯示出在特定日期對這些語言科目知識的高度掌握。\n\n### 常見迷思整理（關於成績與學習）\n\n以下是一些關於成績和學習的常見迷思，與上述成績本身無關，但可作為對一般學業表現的思考：\n\n1.  **迷思一：高分代表完美掌握所有知識點。**\n    *   **說明：** 考試通常只涵蓋課程內容的一部分。即使取得高分，學生可能在某些未考到的細節或更深層次的應用上仍有進步空間。\n\n2.  **迷思二：一旦獲得高分，學習就結束了，無需再精進。**\n    *   **說明：** 學習是一個持續的過程，尤其對於語言科目，知識和技能需要不斷練習、應用和更新才能保持。\n\n3.  **迷思三：高分完全是天賦使然，與努力和策略無關。**\n    *   **說明：** 雖然天賦可能有所幫助，但絕大多數高分都是來自於系統性的學習方法、大量的練習、有效的時間管理以及持之以恆的努力。\n\n4.  **迷思四：考試分數是衡量學習成果的唯一標準。**\n    *   **說明：** 考試分數是評估學習的一種方式，但並非唯一。學習成果也包括實際應用能力、解決問題的能力、批判性思考、學習興趣以及對知識的熱情等。\n\n5.  **迷思五：在特定科目中獲得高分，代表在所有科目或領域都會表現出色。**\n    *   **說明：** 不同科目或領域需要不同的技能、興趣和思維方式。學生可能在某些領域表現卓越，而在其他領域則有不同的學習曲線或興趣點。\n\n---'

In [12]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：德文
請輸入 德文 的成績：99
已記錄：日期: 2026-04-01, 科目: 德文, 成績: 99

請輸入科目（或輸入 'q' 停止）：日語
請輸入 日語 的成績：99
已記錄：日期: 2026-04-01, 科目: 日語, 成績: 99

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理：

---

### 成績摘要

這份成績列表顯示了單一學生在兩個語言科目上的學習表現，所有成績均記錄於2026年4月1日：

*   **科目：** 德文
    *   **成績：** 99分
*   **科目：** 日語
    *   **成績：** 99分

整體而言，該學生在德文和日語兩門科目上均獲得了99分，顯示了高度一致的學習成果。

### 常見迷思整理（關於成績與學習）

在解讀學生成績時，人們常會有一些既定的觀念，以下列出幾個常見的迷思與其說明，以提供更全面的視角：

1.  **迷思一：成績是衡量學習成果的唯一標準。**
    *   **澄清：** 成績固然是評估學生在特定時間點、特定課程內容上表現的重要指標，但它並非學習成果的全部。真正的學習還包括理解力、應用能力、解決問題的能力、批判性思維、創造力以及對學習內容的熱情和好奇心。這些軟實力往往無法完全透過分數來衡量。

2.  **迷思二：高分總是等同於深刻的理解與應用能力。**
    *   **澄清：** 高分通常代表學生掌握了課程內容，但在某些情況下，高分可能來自於良好的應試技巧、記憶力或重複練習，而非對知識的深層理解或靈活運用。真正的理解需要在不同情境下應用知識、連結不同概念，甚至創造新知。

3.  **迷思三：一次成績就能完全定義學生的能力或潛力。**
    *   **澄清：**